# Proyecto 1 · Optimización de funciones (A) — Algoritmo genético binario
**Fundamentos de Inteligencia Artificial** · Basado en *Algoritmos Evolutivos: un enfoque práctico* (Araujo y Cervigón), Proyecto 1, págs. 151–156, y capítulos 2 y 3.

**Alumno:** Myranda Belen Infante Castillo · **Matrícula:** 2430384 · ITIID · UPV

Diseño del algoritmo (tal como lo pide el libro):

| Elemento | Opción |
|---|---|
| Representación | Cadenas binarias (una por variable) |
| Función de evaluación | $f$ evaluada en el punto que resulta de decodificar el individuo |
| Selección | Ruleta |
| Cruce | Monopunto |
| Mutación | Aleatoria bit a bit |
| Mejora | Elitismo (opcional) y revisión de la adaptación para minimizar |

### Cómo usar este cuaderno
1. *Entorno de ejecución → Ejecutar todas* (Ctrl+F9) la primera vez.
2. Para cambiar la función o cualquier parámetro, usa el **formulario de la celda 3** y vuelve a *Ejecutar todas* (o ejecuta de la celda 3 hacia abajo).
3. Las secciones 8–10 muestran los resultados, las tablas y las gráficas de cada generación; la 11 hace el estudio de parámetros.

In [ ]:
#@title 1. Librerías
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

plt.rcParams["animation.embed_limit"] = 80   # MB, para animaciones largas en Colab
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_rows", 200)
print("Librerías cargadas correctamente")

## 2. Catálogo de funciones del libro
Todas las funciones del **Proyecto 1** (que el **Proyecto 2** vuelve a usar). Para cambiar de función solo se elige en el menú de la celda 3.

| Clave | Función | Intervalo | Objetivo | Óptimo (libro) |
|---|---|---|---|---|
| `max_f1` | $\sin^6(5\pi x)\,e^{-2\ln 2\left(\frac{x-0.1}{0.8}\right)^2}$ | $[0,1]$ | máx | 1.0 en $x=0.1$ |
| `max_f2` | $20+e-20e^{-0.2\lvert x\rvert}-e^{\cos 2\pi x}$ | $[0,32]$ | máx | 22.3136 en $x=31.5005$ |
| `max_f3` | $x+\lvert\sin(32\pi x)\rvert$ | $[0,1]$ | máx | 1.98442 en $x=0.98447$ |
| `max_f4` | $0.5-\dfrac{\sin^2(x)-0.5}{1+0.001x^4}$ | $[0,10]$ | máx | 1.0 en $x=0$ |
| `min_f1` | $1+\dfrac{\cos x}{1+0.01x^2}$ | $[0,50]$ | mín | 0.0883634 en $x=3.08531$ |
| `min_f2` | $-\lvert x\sin(\sqrt{\lvert x\rvert})\rvert$ | $[-250,250]$ | mín | −201.843 en $x=\pm 203.814$ |
| `min_f3` | $\dfrac{\sin x}{1+\sqrt{x}+\frac{\cos x}{1+x}}$ | $[0,25]$ | mín | −0.318071 en $x=4.5798$ |
| `min_f4` | $5+e^{0.5\sin(\pi x^2)}-e^{0.1x\cos(\pi x)}$ | $[0,25]$ | mín | −5.39204 en $x=23.9901$ |
| `var_schaffer` | $0.5-\dfrac{\sin^2\sqrt{x_1^2+x_2^2}-0.5}{(1+0.001(x_1^2+x_2^2))^2}$ | $[-40,40]^2$ | máx | 1.0 en $(0,0)$ |
| `var_rastrigin` | $10n+\sum (x_i^2-10\cos 2\pi x_i)$ | $[-5.12,5.12]^n$ | mín | 0 en $x_i=0$ |
| `var_michalewicz` | $-\sum \sin(x_i)\sin^{20}\!\left(\frac{(i+1)x_i^2}{\pi}\right)$ | $[0,\pi]^n$ | mín | tabla del libro (n=2: −1.959091) |
| `var_schwefel` | $\sum -x_i\sin(\sqrt{\lvert x_i\rvert})$ | $[0,100]^n$ | mín | $-n\cdot 63.63498$ en $x_i=65.54785$ |

> **Nota sobre `min_f4`:** en el libro aparece impresa como $5+\exp(0.5(\pi x^2))-\dots$, pero así no puede tener el mínimo que reporta (−5.39204), porque $e^{0.5\pi x^2}$ crece sin límite. Al comprobarlo numéricamente, la versión $5+e^{0.5\sin(\pi x^2)}-e^{0.1x\cos(\pi x)}$ da **exactamente** −5.39204 en $x=23.9901$, así que es la que se usa aquí (falta el "sen" en la impresión).

In [ ]:
#@title 2. Catálogo de funciones (el "ambiente")
PI = np.pi

# Las funciones reciben x como lista [x1, x2, ..., xn] y funcionan tanto con
# números como con arreglos de numpy (eso permite graficarlas rápidamente).

# ---------- Proyecto 1, sección 1: MAXIMIZACIÓN ----------
def max_f1(x):   # función del pizarrón
    x = x[0]
    return np.sin(5 * PI * x) ** 6 * np.exp(-2 * np.log(2) * ((x - 0.1) / 0.8) ** 2)

def max_f2(x):
    x = x[0]
    return 20 + np.e - 20 * np.exp(-0.2 * np.abs(x)) - np.exp(np.cos(2 * PI * x))

def max_f3(x):
    x = x[0]
    return x + np.abs(np.sin(32 * PI * x))

def max_f4(x):
    x = x[0]
    return 0.5 - (np.sin(x) ** 2 - 0.5) / (1 + 0.001 * x ** 4)

# ---------- Proyecto 1, sección 2: MINIMIZACIÓN ----------
def min_f1(x):
    x = x[0]
    return 1 + np.cos(x) / (1 + 0.01 * x ** 2)

def min_f2(x):
    x = x[0]
    return -np.abs(x * np.sin(np.sqrt(np.abs(x))))

def min_f3(x):
    x = x[0]
    return np.sin(x) / (1 + np.sqrt(x) + np.cos(x) / (1 + x))

def min_f4(x):   # corregida (ver nota de arriba)
    x = x[0]
    return 5 + np.exp(0.5 * np.sin(PI * x ** 2)) - np.exp(0.1 * x * np.cos(PI * x))

# ---------- Proyecto 1, sección 3: VARIAS VARIABLES ----------
def var_schaffer(x):
    r2 = x[0] ** 2 + x[1] ** 2
    return 0.5 - (np.sin(np.sqrt(r2)) ** 2 - 0.5) / (1.0 + 0.001 * r2) ** 2

def var_rastrigin(x):
    n = len(x)
    return 10 * n + sum(xi ** 2 - 10 * np.cos(2 * PI * xi) for xi in x)

def var_michalewicz(x):
    return -sum(np.sin(xi) * np.sin((i + 1) * xi ** 2 / PI) ** 20 for i, xi in enumerate(x, start=1))

def var_schwefel(x):
    return sum(-xi * np.sin(np.sqrt(np.abs(xi))) for xi in x)

MICHALEWICZ_MIN = {1: -1, 2: -1.959091, 3: -2.897553, 4: -3.886358, 5: -4.886358,
                   6: -5.879585, 7: -6.862457, 8: -7.858851, 9: -8.858851, 10: -9.856182}

# n = None significa "número de variables configurable"
FUNCIONES = {
    "max_f1": dict(f=max_f1, objetivo="max", n=1, limites=(0.0, 1.0),
                   formula="sen⁶(5πx)·e^(−2ln2((x−0.1)/0.8)²)", x_opt=[[0.1]], f_opt=1.0),
    "max_f2": dict(f=max_f2, objetivo="max", n=1, limites=(0.0, 32.0),
                   formula="20 + e − 20e^(−0.2|x|) − e^(cos 2πx)", x_opt=[[31.5005]], f_opt=22.3136),
    "max_f3": dict(f=max_f3, objetivo="max", n=1, limites=(0.0, 1.0),
                   formula="x + |sen(32πx)|", x_opt=[[0.98447395]], f_opt=1.98442447),
    "max_f4": dict(f=max_f4, objetivo="max", n=1, limites=(0.0, 10.0),
                   formula="0.5 − (sen²x − 0.5)/(1 + 0.001x⁴)", x_opt=[[0.0]], f_opt=1.0),
    "min_f1": dict(f=min_f1, objetivo="min", n=1, limites=(0.0, 50.0),
                   formula="1 + cos x/(1 + 0.01x²)", x_opt=[[3.08531]], f_opt=0.0883634),
    "min_f2": dict(f=min_f2, objetivo="min", n=1, limites=(-250.0, 250.0),
                   formula="−|x·sen(√|x|)|", x_opt=[[203.814], [-203.814]], f_opt=-201.843),
    "min_f3": dict(f=min_f3, objetivo="min", n=1, limites=(0.0, 25.0),
                   formula="sen x/(1 + √x + cos x/(1+x))", x_opt=[[4.5798]], f_opt=-0.318071),
    "min_f4": dict(f=min_f4, objetivo="min", n=1, limites=(0.0, 25.0),
                   formula="5 + e^(0.5·sen(πx²)) − e^(0.1x·cos(πx))", x_opt=[[23.9901]], f_opt=-5.39204),
    "var_schaffer": dict(f=var_schaffer, objetivo="max", n=2, limites=(-40.0, 40.0),
                   formula="0.5 − (sen²√(x₁²+x₂²) − 0.5)/(1 + 0.001(x₁²+x₂²))²", x_opt=lambda n: [[0.0, 0.0]], f_opt=1.0),
    "var_rastrigin": dict(f=var_rastrigin, objetivo="min", n=None, limites=(-5.12, 5.12),
                   formula="10n + Σ(xᵢ² − 10cos 2πxᵢ)", x_opt=lambda n: [[0.0] * n], f_opt=lambda n: 0.0),
    "var_michalewicz": dict(f=var_michalewicz, objetivo="min", n=None, limites=(0.0, PI),
                   formula="−Σ sen(xᵢ)·sen²⁰((i+1)xᵢ²/π)", x_opt=lambda n: None, f_opt=lambda n: MICHALEWICZ_MIN.get(n)),
    "var_schwefel": dict(f=var_schwefel, objetivo="min", n=None, limites=(0.0, 100.0),
                   formula="Σ −xᵢ·sen(√|xᵢ|)", x_opt=lambda n: [[65.54785] * n], f_opt=lambda n: -n * 63.63498),
}

def preparar_funcion(nombre, n_variables=2):
    """Devuelve la función lista para usarse: número de variables, límites de cada una y óptimo."""
    d = FUNCIONES[nombre]
    n = d["n"] if d["n"] is not None else int(n_variables)
    x_opt = d["x_opt"](n) if callable(d["x_opt"]) else d["x_opt"]
    f_opt = d["f_opt"](n) if callable(d["f_opt"]) else d["f_opt"]
    return dict(nombre=nombre, f=d["f"], objetivo=d["objetivo"], n=n,
                limites=[d["limites"]] * n, formula=d["formula"], x_opt=x_opt, f_opt=f_opt)

pd.DataFrame([{"clave": k, "objetivo": v["objetivo"], "variables": v["n"] or "n",
               "intervalo": v["limites"], "fórmula": v["formula"]} for k, v in FUNCIONES.items()])

## 3. Parámetros (formulario)
Todo lo que pida el doctor se cambia aquí. Los parámetros son los que el libro indica para el estudio: precisión, tamaño de población, número de generaciones, porcentaje de cruces y porcentaje de mutaciones.
- `N_VARIABLES` solo se usa con `var_rastrigin`, `var_michalewicz` y `var_schwefel` (con 2 variables se ven los individuos moverse sobre el mapa de la función).
- `PROB_MUT` es la probabilidad de mutar **cada bit**.
- `SEMILLA = -1` da resultados distintos cada vez; cualquier otro número hace el experimento repetible.

In [ ]:
#@title 3. Parámetros del algoritmo genético (Proyecto 1)
FUNCION = "max_f1" #@param ["max_f1", "max_f2", "max_f3", "max_f4", "min_f1", "min_f2", "min_f3", "min_f4", "var_schaffer", "var_rastrigin", "var_michalewicz", "var_schwefel"]
N_VARIABLES = 2 #@param {type:"integer"}
PREC = 0.001 #@param {type:"number"}
TAM_POB = 100 #@param {type:"integer"}
NUM_GENERACIONES = 100 #@param {type:"integer"}
PROB_CRUCE = 0.6 #@param {type:"slider", min:0, max:1, step:0.05}
PROB_MUT = 0.02 #@param {type:"slider", min:0, max:0.2, step:0.005}
ELITISMO = True #@param {type:"boolean"}
PORC_ELITE = 0.02 #@param {type:"number"}
SEMILLA = 42 #@param {type:"integer"}

CONFIG = dict(FUNCION=FUNCION, N_VARIABLES=N_VARIABLES, PREC=PREC, TAM_POB=TAM_POB,
              NUM_GENERACIONES=NUM_GENERACIONES, PROB_CRUCE=PROB_CRUCE, PROB_MUT=PROB_MUT,
              ELITISMO=ELITISMO, PORC_ELITE=PORC_ELITE, SEMILLA=SEMILLA)
FUNC = preparar_funcion(FUNCION, N_VARIABLES)
print("Función:", FUNC["nombre"], "→", FUNC["formula"], f"({FUNC['objetivo']}, {FUNC['n']} variable(s))")
CONFIG

## 4. Longitud del cromosoma (sección 2.10 del libro)
Se divide el intervalo en porciones de tamaño menor que PREC:

$$l = \left\lceil \log_2\!\left(1 + \frac{x_{max} - x_{min}}{PREC}\right) \right\rceil$$

Ejemplos:
- Pizarrón, $[0,1]$ con $PREC=0.001$: $\log_2(1001) = 9.967 \Rightarrow l = 10$ bits.
- Apuntes de clase, $[-10,10]$: $\log_2(20001) = 14.2878 \Rightarrow l = 15$ bits.
- Ejemplo del libro, $[0,20]$ con $PREC=0.0001$: $l = 18$ bits.

In [ ]:
#@title 4. Longitud del cromosoma
def longitud_cromosoma(x_min, x_max, prec):
    return math.ceil(math.log2(1 + (x_max - x_min) / prec))

for (a, b), i in zip(FUNC["limites"], range(1, FUNC["n"] + 1)):
    valor = math.log2(1 + (b - a) / PREC)
    l = longitud_cromosoma(a, b, PREC)
    print(f"x{i} ∈ [{a}, {b}]  →  log2 = {valor:.4f}  →  l = {l} bits  "
          f"(2^{l} = {2**l} valores, porción real = {(b - a) / (2**l - 1):.6f})")

# Comprobación de los ejemplos de clase y del libro
print("Clase  [-10, 10], PREC 0.001  →", longitud_cromosoma(-10, 10, 0.001), "bits")
print("Libro  [0, 20],   PREC 0.0001 →", longitud_cromosoma(0, 20, 0.0001), "bits")

## 5. Clase `Individuo` (genotipo → fenotipo → adaptación)
Como se vio en clase:
- `gen` — **genotipo**: la cadena binaria (una por variable), ej. `1011101101`.
- `x` — **fenotipo**: el valor real que representa la cadena.
- `adapt` — **adaptación**: la salida de la función, $f(x)$.

Además, el libro (2.9.1) agrega `aptitud` (adaptación revisada), `puntuacion` y `punt_acu` para la ruleta.

Decodificación (2.9.3): $x = x_{min} + \text{bin\_ent}(c)\cdot\dfrac{x_{max}-x_{min}}{2^l-1}$

In [ ]:
#@title 5. Clase Individuo (representación binaria)
class Individuo:
    def __init__(self, problema, gen=None):
        self.problema = problema
        if gen is None:   # genera_indiv (2.9.2): cada bit es 0 o 1 al azar
            gen = [[random.randint(0, 1) for _ in range(l)] for l in problema["lcrom"]]
        self.gen = gen            # genotipo: una cadena binaria por variable
        self.x = None             # fenotipo: valor real de cada variable
        self.adapt = None         # adaptación: f(x)
        self.aptitud = 0.0        # adaptación revisada (positiva) que usa la ruleta
        self.puntuacion = 0.0     # aptitud / suma de aptitudes
        self.punt_acu = 0.0       # puntuación acumulada para los sorteos
        self.evaluar()

    @staticmethod
    def bin_ent(cadena):
        """Convierte la cadena binaria en entero (2.9.3)."""
        d = 0
        for bit in cadena:
            d = d * 2 + bit
        return d

    def decodificar(self):
        self.x = [a + self.bin_ent(c) * (b - a) / (2 ** len(c) - 1)
                  for c, (a, b) in zip(self.gen, self.problema["limites"])]
        return self.x

    def evaluar(self):
        self.decodificar()
        self.adapt = float(self.problema["f"](self.x))
        return self.adapt

    def copia(self):
        return Individuo(self.problema, [c[:] for c in self.gen])

    def cadena(self):
        return " ".join("".join(map(str, c)) for c in self.gen)

    def __repr__(self):
        return f"gen={self.cadena()} | x={np.round(self.x, 6).tolist()} | adapt={self.adapt:.6f}"

# Prueba rápida con la función seleccionada
random.seed(1)
_prob = {**FUNC, "lcrom": [longitud_cromosoma(a, b, PREC) for a, b in FUNC["limites"]]}
for _ in range(3):
    print(Individuo(_prob))

## 6. Operadores comunes (capítulos 2 y 3 del libro)
- **Revisión de la adaptación** (sección 3.1.1): la ruleta necesita valores positivos y que "mejor" signifique "más grande".
  - Maximizar: `aptitud = f(x)` (si hubiera valores negativos se desplazan para que sean positivos).
  - Minimizar: `aptitud = cmax·1.05 − f(x)`, donde `cmax` es el mayor valor de la población. El 5 % extra evita que la suma sea 0 si toda la población converge. *(Se escribe como `cmax + 0.05·|cmax|`: es lo mismo que `cmax·1.05` cuando `cmax > 0`, y sigue funcionando cuando `cmax` es negativo, por ejemplo en `min_f2`.)*
- **Evaluación** (2.9.4): calcula `puntuacion = aptitud / sumadaptacion` y la `punt_acu` acumulada para los sorteos.
- **Selección por ruleta** (2.9.5): se generan `tam_pob` números aleatorios y cada uno elige al primer individuo cuya `punt_acu` lo supera.
- **Reproducción** (2.9.6): cada individuo entra al cruce con probabilidad `PROB_CRUCE`; los elegidos se cruzan por parejas y **los hijos sustituyen a los padres**.
- **Elitismo** (3.2): se reservan los mejores `E` individuos (1–2 % de la población) y, al final de la generación, reemplazan a los peores.

In [ ]:
#@title 6. Operadores comunes: adaptación, evaluación, ruleta, reproducción y elitismo
def es_mejor(a, b, objetivo):
    """True si el valor a es mejor que b según se maximice o minimice."""
    return a > b if objetivo == "max" else a < b

def revisar_adaptacion(pob, objetivo):
    """Sección 3.1.1: convierte f(x) en una aptitud positiva para la ruleta."""
    valores = [ind.adapt for ind in pob]
    if objetivo == "min":
        cmax = max(valores)
        cmax = cmax + 0.05 * abs(cmax)          # = cmax*1.05 cuando cmax > 0
        for ind in pob:
            ind.aptitud = cmax - ind.adapt
    else:
        fmin = min(valores)
        base = fmin - 0.05 * abs(fmin) if fmin < 0 else 0.0
        for ind in pob:
            ind.aptitud = ind.adapt - base

def evaluacion(pob, objetivo):
    """Sección 2.9.4: puntuación relativa, puntuación acumulada y posición del mejor."""
    sumadap = sum(ind.aptitud for ind in pob)
    pos_mejor = 0
    for i, ind in enumerate(pob):
        if es_mejor(ind.adapt, pob[pos_mejor].adapt, objetivo):
            pos_mejor = i
    punt_acu = 0.0
    for ind in pob:
        ind.puntuacion = ind.aptitud / sumadap if sumadap > 0 else 1.0 / len(pob)
        punt_acu += ind.puntuacion
        ind.punt_acu = punt_acu
    pob[-1].punt_acu = 1.0                      # evita errores de redondeo en el último
    return pos_mejor, sumadap

def seleccion_ruleta(pob):
    """Sección 2.9.5: selecciona tam_pob supervivientes por el método de la ruleta."""
    nueva = []
    for _ in range(len(pob)):
        prob = random.random()
        pos_super = 0
        while prob > pob[pos_super].punt_acu and pos_super < len(pob) - 1:
            pos_super += 1
        nueva.append(pob[pos_super].copia())    # copia: cada superviviente es independiente
    return nueva

def reproduccion(pob, prob_cruce, cruce):
    """Sección 2.9.6: elige con probabilidad prob_cruce, cruza por parejas y los hijos sustituyen a los padres."""
    sel_cruce = [i for i in range(len(pob)) if random.random() < prob_cruce]
    if len(sel_cruce) % 2 == 1:                 # el número de seleccionados se hace par
        sel_cruce.pop()
    for k in range(0, len(sel_cruce), 2):
        a, b = sel_cruce[k], sel_cruce[k + 1]
        pob[a], pob[b] = cruce(pob[a], pob[b])

def obtener_elite(pob, tam_elite, objetivo):
    """Sección 3.2: copias de los tam_elite mejores individuos."""
    if tam_elite <= 0:
        return []
    ordenados = sorted(pob, key=lambda ind: ind.adapt, reverse=(objetivo == "max"))
    return [ind.copia() for ind in ordenados[:tam_elite]]

def incluir_elite(pob, elite, objetivo):
    """La élite reemplaza a los peores individuos de la nueva generación."""
    peores = sorted(range(len(pob)), key=lambda i: pob[i].adapt, reverse=(objetivo == "min"))
    for pos, ind in zip(peores, elite):
        pob[pos] = ind

def registrar(pob, generacion, pos_mejor):
    """Guarda una "foto" de la generación para las tablas y las gráficas."""
    F = np.array([ind.adapt for ind in pob])
    return dict(generacion=generacion,
                X=np.array([ind.x for ind in pob]),          # fenotipos (pob × n)
                F=F,                                         # adaptaciones f(x)
                genotipos=[ind.cadena() for ind in pob],
                mejor_x=list(pob[pos_mejor].x),
                mejor_f=float(pob[pos_mejor].adapt),
                media=float(F.mean()))

def generacion_optimo(res, tol=None):
    """Primera generación en la que el mejor individuo alcanza el óptimo teórico (± tolerancia)."""
    f_opt = res["func"]["f_opt"]
    if f_opt is None:
        return None
    tol = tol if tol is not None else 1e-3 * max(1.0, abs(f_opt))
    for h in res["historial"]:
        if abs(h["mejor_f"] - f_opt) <= tol:
            return h["generacion"]
    return None

def imprimir_resumen(res):
    func, mejor = res["func"], res["mejor"]
    print("=" * 70)
    print(f"Función : {func['nombre']}   →   f = {func['formula']}")
    print(f"Objetivo: {'MAXIMIZAR' if func['objetivo'] == 'max' else 'MINIMIZAR'}   |   variables: {func['n']}")
    print("-" * 70)
    print(f"Mejor individuo : {mejor.cadena()}")
    print(f"x (fenotipo)    : {np.round(mejor.x, 6).tolist()}")
    print(f"f(x) (adapt)    : {mejor.adapt:.6f}")
    if func["f_opt"] is not None:
        print(f"Óptimo del libro: {func['f_opt']}" + (f"  en x = {func['x_opt']}" if func["x_opt"] else ""))
        print(f"Error absoluto  : {abs(mejor.adapt - func['f_opt']):.6f}")
        g = generacion_optimo(res)
        print("Óptimo alcanzado en la generación: " + (str(g) if g is not None else "no se alcanzó (±0.1 %)"))
    print("=" * 70)

def tabla_poblacion(res, generacion=0):
    """Tabla como las del libro (sección 2.10): posición, individuo, x y adaptación."""
    h = res["historial"][generacion]
    return pd.DataFrame({"Posición": range(1, len(h["F"]) + 1),
                         "Individuo": h["genotipos"],
                         "x": [np.round(x, 6).tolist() if len(x) > 1 else round(float(x[0]), 6) for x in h["X"]],
                         "adaptación": np.round(h["F"], 6)})

def tabla_generaciones(res):
    """Evolución de la adaptación media y máxima (mínima) por generación."""
    return pd.DataFrame([{"Generación": h["generacion"], "Adap. media": round(h["media"], 6),
                          "Mejor adap.": round(h["mejor_f"], 6),
                          "x del mejor": np.round(h["mejor_x"], 6).tolist()} for h in res["historial"]])

## 7. Cruce monopunto, mutación bit a bit y el algoritmo genético completo
- **Cruce monopunto** (2.7.1 / 2.9.6): se elige un punto al azar y se intercambian los segmentos. Con varias variables se cruza $x_1$ del padre 1 con $x_1$ del padre 2, $x_2$ con $x_2$, etc. (sección 3 del proyecto).
- **Mutación bit a bit** (2.7.2): cada bit cambia con probabilidad `PROB_MUT`; si el individuo mutó se vuelve a evaluar.
- **Bucle** (2.9): selección → reproducción → mutación → (elitismo) → revisión de la adaptación → evaluación, hasta `NUM_GENERACIONES`.

In [ ]:
#@title 7. Cruce, mutación y algoritmo genético (Proyecto 1)
def cruce_monopunto(padre1, padre2):
    hijo1, hijo2 = [], []
    for c1, c2 in zip(padre1.gen, padre2.gen):          # x1 con x1, x2 con x2, ...
        punto = random.randint(1, len(c1) - 1) if len(c1) > 1 else 1
        hijo1.append(c1[:punto] + c2[punto:])
        hijo2.append(c2[:punto] + c1[punto:])
    return Individuo(padre1.problema, hijo1), Individuo(padre1.problema, hijo2)

def mutacion(pob, prob_mut):
    for ind in pob:
        mutado = False
        for cadena in ind.gen:
            for j in range(len(cadena)):
                if random.random() < prob_mut:
                    cadena[j] = 1 - cadena[j]               # se invierte el bit
                    mutado = True
        if mutado:
            ind.evaluar()

def algoritmo_genetico(cfg=None, verbose=True):
    cfg = cfg or CONFIG
    if cfg["SEMILLA"] is not None and cfg["SEMILLA"] >= 0:
        random.seed(cfg["SEMILLA"])
        np.random.seed(cfg["SEMILLA"])
    func = preparar_funcion(cfg["FUNCION"], cfg["N_VARIABLES"])
    objetivo = func["objetivo"]
    problema = {**func, "lcrom": [longitud_cromosoma(a, b, cfg["PREC"]) for a, b in func["limites"]]}
    tam_elite = max(1, round(cfg["PORC_ELITE"] * cfg["TAM_POB"])) if cfg["ELITISMO"] else 0

    pob = [Individuo(problema) for _ in range(cfg["TAM_POB"])]        # población inicial
    revisar_adaptacion(pob, objetivo)
    pos_mejor, _ = evaluacion(pob, objetivo)
    mejor_global = pob[pos_mejor].copia()
    historial = [registrar(pob, 0, pos_mejor)]

    for g in range(1, cfg["NUM_GENERACIONES"] + 1):                  # bucle de evolución
        elite = obtener_elite(pob, tam_elite, objetivo)
        pob = seleccion_ruleta(pob)
        reproduccion(pob, cfg["PROB_CRUCE"], cruce_monopunto)
        mutacion(pob, cfg["PROB_MUT"])
        incluir_elite(pob, elite, objetivo)
        revisar_adaptacion(pob, objetivo)
        pos_mejor, _ = evaluacion(pob, objetivo)
        if es_mejor(pob[pos_mejor].adapt, mejor_global.adapt, objetivo):
            mejor_global = pob[pos_mejor].copia()
        historial.append(registrar(pob, g, pos_mejor))

    res = dict(mejor=mejor_global, poblacion=pob, historial=historial, func=func,
               cfg=dict(cfg), lcrom=problema["lcrom"], tam_elite=tam_elite)
    if verbose:
        imprimir_resumen(res)
    return res

## 8. Ejecución y resultados

In [ ]:
#@title 8. Ejecutar el algoritmo genético
res = algoritmo_genetico(CONFIG)
print(f"Longitud del cromosoma: {res['lcrom']} bits  |  tamaño de la élite: {res['tam_elite']}")

In [ ]:
#@title 8.1 Población inicial (tabla como la del libro)
tabla_poblacion(res, generacion=0)

In [ ]:
#@title 8.2 Población final
tabla_poblacion(res, generacion=len(res["historial"]) - 1)

In [ ]:
#@title 8.3 Adaptación media y mejor por generación
tabla_generaciones(res)

## 9. Gráficas: la función y cómo se mueven los individuos en cada generación
Funciones de graficación (se usan en la siguiente sección):
- `graficar_funcion` — la función sola, con el óptimo teórico marcado.
- `graficar_generaciones` — varias generaciones lado a lado sobre la función.
- `graficar_cada_generacion` — una gráfica por cada generación (cada *k* generaciones).
- `animar_generaciones` — **animación con barra de reproducción** que recorre todas las generaciones; opcionalmente guarda un GIF.
- `graficar_convergencia` — mejor adaptación y adaptación media por generación (lo que pide el libro).
- `graficar_trayectorias` — posición de cada individuo (cada variable) a lo largo de las generaciones; sirve para cualquier número de variables.

Con 1 variable los individuos se dibujan sobre la curva $f(x)$; con 2 variables sobre las curvas de nivel de $f(x_1,x_2)$. Con más de 2 variables se usan las trayectorias.

In [ ]:
#@title 9. Funciones de graficación
COLOR_FUNCION = "#6c757d"   # curva / mapa de la función (neutro)
COLOR_INDIV   = "#0072B2"   # individuos de la población
COLOR_MEJOR   = "#D55E00"   # mejor individuo de la generación
COLOR_MEDIA   = "#E69F00"   # adaptación media
PALETA = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00"]  # Okabe-Ito (apta para daltonismo)

def _titulo_funcion(func):
    return f"{func['nombre']}: f = {func['formula']}  ({'máx' if func['objetivo'] == 'max' else 'mín'})"

_cache_malla = {}
def _malla_2d(func, puntos=250):
    clave = (func["nombre"], func["n"])
    if clave not in _cache_malla:
        (a1, b1), (a2, b2) = func["limites"][0], func["limites"][1]
        X1, X2 = np.meshgrid(np.linspace(a1, b1, puntos), np.linspace(a2, b2, puntos))
        if func["n"] == 2:
            Z = func["f"]([X1, X2])
        else:   # más de 2 variables: corte en el plano x1-x2 con el resto en su óptimo (o el centro)
            resto = (func["x_opt"][0][2:] if func["x_opt"] else
                     [(a + b) / 2 for a, b in func["limites"][2:]])
            Z = func["f"]([X1, X2] + [np.full_like(X1, v) for v in resto])
        _cache_malla[clave] = (X1, X2, Z)
    return _cache_malla[clave]

def _dibujar_fondo(ax, func, barra=True):
    if func["n"] == 1:
        a, b = func["limites"][0]
        xs = np.linspace(a, b, 3000)
        ax.plot(xs, func["f"]([xs]), color=COLOR_FUNCION, lw=1.4, label="f(x)", zorder=1)
        ax.set_xlabel("x (fenotipo)")
        ax.set_ylabel("f(x) (adaptación)")
    else:
        X1, X2, Z = _malla_2d(func)
        cs = ax.contourf(X1, X2, Z, levels=30, cmap="Greys", alpha=0.85, zorder=0)
        if barra:
            plt.colorbar(cs, ax=ax, label="f(x₁, x₂)")
        ax.set_xlabel("x₁")
        ax.set_ylabel("x₂")
    ax.grid(True, alpha=0.25)

def _marcar_optimo(ax, func):
    if not func["x_opt"]:
        return
    for k, xo in enumerate(func["x_opt"]):
        lbl = "Óptimo teórico" if k == 0 else None
        if func["n"] == 1:
            ax.scatter(xo[0], func["f_opt"], marker="X", s=90, color="black", zorder=2, label=lbl)
        else:
            ax.scatter(xo[0], xo[1], marker="X", s=90, color="black", zorder=2, label=lbl)

def _dibujar_poblacion(ax, h, func):
    if func["n"] == 1:
        p = ax.scatter(h["X"][:, 0], h["F"], s=36, color=COLOR_INDIV, edgecolors="white",
                       linewidths=0.8, zorder=3, label="Individuos")
        m = ax.scatter([h["mejor_x"][0]], [h["mejor_f"]], s=220, marker="*", color=COLOR_MEJOR,
                       edgecolors="white", linewidths=0.8, zorder=4, label="Mejor")
    else:
        p = ax.scatter(h["X"][:, 0], h["X"][:, 1], s=36, color=COLOR_INDIV, edgecolors="white",
                       linewidths=0.8, zorder=3, label="Individuos")
        m = ax.scatter([h["mejor_x"][0]], [h["mejor_x"][1]], s=220, marker="*", color=COLOR_MEJOR,
                       edgecolors="white", linewidths=0.8, zorder=4, label="Mejor")
    return p, m

def graficar_funcion(func):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    _dibujar_fondo(ax, func)
    _marcar_optimo(ax, func)
    ax.set_title(_titulo_funcion(func))
    if func["x_opt"]:
        ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

def graficar_generaciones(res, paneles=6, columnas=3):
    """Varias generaciones lado a lado (generación 0, intermedias y la última)."""
    func, hist = res["func"], res["historial"]
    gens = sorted(set(np.linspace(0, len(hist) - 1, paneles).round().astype(int)))
    filas = math.ceil(len(gens) / columnas)
    fig, axes = plt.subplots(filas, columnas, figsize=(5.2 * columnas, 3.8 * filas), squeeze=False)
    for ax, g in zip(axes.flat, gens):
        h = hist[g]
        _dibujar_fondo(ax, func, barra=False)
        _marcar_optimo(ax, func)
        _dibujar_poblacion(ax, h, func)
        ax.set_title(f"Generación {g}  ·  mejor f = {h['mejor_f']:.5f}", fontsize=10)
    for ax in list(axes.flat)[len(gens):]:
        ax.axis("off")
    axes.flat[0].legend(loc="best", fontsize=8)
    fig.suptitle(_titulo_funcion(func), fontsize=12)
    plt.tight_layout()
    plt.show()

def graficar_cada_generacion(res, cada=1, maximo=200):
    """Una gráfica por generación, cada `cada` generaciones (incluye siempre la última)."""
    func, hist = res["func"], res["historial"]
    gens = list(range(0, len(hist), max(1, cada)))
    if gens[-1] != len(hist) - 1:
        gens.append(len(hist) - 1)
    for g in gens[:maximo]:
        h = hist[g]
        fig, ax = plt.subplots(figsize=(8, 3.6))
        _dibujar_fondo(ax, func)
        _marcar_optimo(ax, func)
        _dibujar_poblacion(ax, h, func)
        ax.set_title(f"Generación {g}  ·  mejor f = {h['mejor_f']:.6f}  ·  media = {h['media']:.6f}")
        ax.legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()

def animar_generaciones(res, paso=1, intervalo_ms=300, guardar_gif=False, nombre_gif="evolucion.gif"):
    """Animación (con controles de reproducción) de los individuos generación por generación."""
    func, hist = res["func"], res["historial"]
    frames = list(range(0, len(hist), max(1, paso)))
    if frames[-1] != len(hist) - 1:
        frames.append(len(hist) - 1)
    fig, ax = plt.subplots(figsize=(9, 4.8), dpi=80)
    _dibujar_fondo(ax, func)
    _marcar_optimo(ax, func)
    puntos, mejor = _dibujar_poblacion(ax, hist[0], func)
    ax.legend(loc="best", fontsize=8)
    titulo = ax.set_title("")

    def actualizar(k):
        h = hist[frames[k]]
        if func["n"] == 1:
            puntos.set_offsets(np.column_stack([h["X"][:, 0], h["F"]]))
            mejor.set_offsets([[h["mejor_x"][0], h["mejor_f"]]])
        else:
            puntos.set_offsets(h["X"][:, :2])
            mejor.set_offsets([h["mejor_x"][:2]])
        titulo.set_text(f"Generación {h['generacion']}  ·  mejor f = {h['mejor_f']:.6f}  ·  media = {h['media']:.6f}")
        return puntos, mejor, titulo

    actualizar(0)
    anim = FuncAnimation(fig, actualizar, frames=len(frames), interval=intervalo_ms, blit=False)
    plt.close(fig)
    if guardar_gif:
        anim.save(nombre_gif, writer=PillowWriter(fps=max(1, round(1000 / intervalo_ms))))
        print(f"GIF guardado como '{nombre_gif}' (panel de archivos de la izquierda en Colab).")
    return HTML(anim.to_jshtml())

def graficar_convergencia(res):
    """Mejor adaptación y adaptación media por generación (sección 1 del Proyecto 1)."""
    func, hist = res["func"], res["historial"]
    g = [h["generacion"] for h in hist]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(g, [h["mejor_f"] for h in hist], color=COLOR_INDIV, lw=2, label="Mejor de la generación")
    ax.plot(g, [h["media"] for h in hist], color=COLOR_MEDIA, lw=2, label="Media de la población")
    if func["f_opt"] is not None:
        ax.axhline(func["f_opt"], color="black", ls="--", lw=1, label=f"Óptimo del libro ({func['f_opt']})")
    ax.set_xlabel("Generación")
    ax.set_ylabel("f(x)")
    ax.set_title("Evolución de la adaptación · " + _titulo_funcion(func), fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

def graficar_trayectorias(res, max_variables=4):
    """Cada punto es un individuo: muestra cómo se mueve cada variable de generación en generación."""
    func, hist = res["func"], res["historial"]
    nv = min(func["n"], max_variables)
    fig, axes = plt.subplots(nv, 1, figsize=(10, 3.0 * nv), sharex=True, squeeze=False)
    for i, ax in enumerate(axes[:, 0]):
        for h in hist:
            ax.scatter(np.full(len(h["X"]), h["generacion"]), h["X"][:, i], s=6,
                       color=COLOR_INDIV, alpha=0.35, linewidths=0)
        ax.plot([h["generacion"] for h in hist], [h["mejor_x"][i] for h in hist],
                color=COLOR_MEJOR, lw=2, label="Mejor individuo")
        if func["x_opt"]:
            for k, xo in enumerate(func["x_opt"]):
                ax.axhline(xo[i], color="black", ls="--", lw=1, label="Óptimo teórico" if k == 0 else None)
        ax.set_ylabel(f"x{i + 1}" if func["n"] > 1 else "x")
        ax.set_ylim(*func["limites"][i])
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right", fontsize=8)
    axes[-1, 0].set_xlabel("Generación")
    fig.suptitle("Movimiento de los individuos por generación · " + func["nombre"], fontsize=11)
    plt.tight_layout()
    plt.show()

## 10. Gráficas de los resultados

In [ ]:
#@title 10.1 La función y su óptimo
if FUNC["n"] <= 2:
    graficar_funcion(FUNC)
else:
    print("Con más de 2 variables la función no se puede dibujar completa; usa las trayectorias (10.5).")

In [ ]:
#@title 10.2 Individuos en varias generaciones
PANELES = 6 #@param {type:"integer"}
if FUNC["n"] <= 2:
    graficar_generaciones(res, paneles=PANELES)
else:
    print("Con más de 2 variables usa las trayectorias (10.5).")

In [ ]:
#@title 10.3 Animación de TODAS las generaciones (usa ▶ para reproducir)
PASO = 1 #@param {type:"integer"}
GUARDAR_GIF = False #@param {type:"boolean"}
if FUNC["n"] <= 2:
    display(animar_generaciones(res, paso=PASO, guardar_gif=GUARDAR_GIF, nombre_gif="proyecto1_evolucion.gif"))
else:
    print("Con más de 2 variables usa las trayectorias (10.5).")

In [ ]:
#@title 10.4 Una gráfica por generación
CADA = 10 #@param {type:"integer"}
# CADA = 1 dibuja absolutamente todas las generaciones (son muchas gráficas).
if FUNC["n"] <= 2:
    graficar_cada_generacion(res, cada=CADA)
else:
    print("Con más de 2 variables usa las trayectorias (10.5).")

In [ ]:
#@title 10.5 Movimiento de los individuos (sirve para cualquier número de variables)
graficar_trayectorias(res)

In [ ]:
#@title 10.6 Convergencia: mejor y media por generación
graficar_convergencia(res)

## 11. Estudio de parámetros (lo pide el libro)
> *"Fijamos todos los parámetros menos uno que vamos variando para realizar el estudio."* — Proyecto 1, sección 1.

`estudio_parametro(nombre, valores, repeticiones)` ejecuta el AG varias veces (con semillas distintas) para cada valor del parámetro, dejando todo lo demás como está en la celda 3. Devuelve una tabla con el promedio del mejor resultado, su desviación, el error frente al óptimo, en cuántas ejecuciones se alcanzó y en qué generación (promedio), y grafica la convergencia promedio de cada valor.

`comparar_elitismo(repeticiones)` compara el AG con y sin elitismo, como sugiere el libro.

In [ ]:
#@title 11. Funciones del estudio de parámetros
def _curva_promedio(resultados):
    return np.mean([[h["mejor_f"] for h in r["historial"]] for r in resultados], axis=0)

def _resumen_ejecuciones(resultados, etiqueta, valor):
    func = resultados[0]["func"]
    mejores = np.array([r["mejor"].adapt for r in resultados])
    gens = [generacion_optimo(r) for r in resultados]
    alcanzadas = [g for g in gens if g is not None]
    fila = {etiqueta: valor,
            "mejor f (promedio)": round(mejores.mean(), 6),
            "desv. estándar": round(mejores.std(), 6),
            "mejor f (de todas)": round(mejores.max() if func["objetivo"] == "max" else mejores.min(), 6)}
    if func["f_opt"] is not None:
        fila["error promedio"] = round(np.abs(mejores - func["f_opt"]).mean(), 6)
        fila["alcanzó el óptimo"] = f"{len(alcanzadas)}/{len(resultados)}"
        fila["generación promedio"] = round(np.mean(alcanzadas), 1) if alcanzadas else "—"
    return fila

def _graficar_curvas(curvas, titulo, func):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for k, (etiqueta, curva) in enumerate(curvas.items()):
        ax.plot(curva, lw=2, color=PALETA[k % len(PALETA)], label=str(etiqueta))
    if func["f_opt"] is not None:
        ax.axhline(func["f_opt"], color="black", ls="--", lw=1, label="Óptimo del libro")
    ax.set_xlabel("Generación")
    ax.set_ylabel("Mejor f(x) (promedio de ejecuciones)")
    ax.set_title(titulo, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()

def estudio_parametro(nombre, valores, repeticiones=5):
    """Varía UN parámetro de CONFIG y deja los demás fijos."""
    filas, curvas = [], {}
    for v in valores:
        resultados = []
        for r in range(repeticiones):
            cfg = dict(CONFIG)
            cfg[nombre] = v
            cfg["SEMILLA"] = r
            resultados.append(algoritmo_genetico(cfg, verbose=False))
        filas.append(_resumen_ejecuciones(resultados, nombre, v))
        curvas[f"{nombre} = {v}"] = _curva_promedio(resultados)
    _graficar_curvas(curvas, f"Estudio de {nombre} · {CONFIG['FUNCION']} · {repeticiones} ejecuciones por valor",
                     resultados[0]["func"])
    return pd.DataFrame(filas)

def comparar_elitismo(repeticiones=10):
    """Mismo AG con y sin elitismo (sección 1 del Proyecto 1)."""
    filas, curvas = [], {}
    for elit in (False, True):
        resultados = []
        for r in range(repeticiones):
            cfg = dict(CONFIG)
            cfg["ELITISMO"] = elit
            cfg["SEMILLA"] = r
            resultados.append(algoritmo_genetico(cfg, verbose=False))
        etiqueta = "Con elitismo" if elit else "Sin elitismo"
        filas.append(_resumen_ejecuciones(resultados, "variante", etiqueta))
        curvas[etiqueta] = _curva_promedio(resultados)
    _graficar_curvas(curvas, f"Con vs. sin elitismo · {CONFIG['FUNCION']} · {repeticiones} ejecuciones",
                     resultados[0]["func"])
    return pd.DataFrame(filas)

In [ ]:
#@title 11.1 Estudio del tamaño de población
estudio_parametro("TAM_POB", [10, 30, 50, 100, 200], repeticiones=5)

In [ ]:
#@title 11.2 Estudio de la probabilidad de cruce
estudio_parametro("PROB_CRUCE", [0.2, 0.4, 0.6, 0.8, 1.0], repeticiones=5)

In [ ]:
#@title 11.3 Estudio de la probabilidad de mutación
estudio_parametro("PROB_MUT", [0.001, 0.01, 0.02, 0.05, 0.1], repeticiones=5)

In [ ]:
#@title 11.4 Estudio de la precisión (error permitido en la discretización)
estudio_parametro("PREC", [0.1, 0.01, 0.001, 0.0001], repeticiones=5)

In [ ]:
#@title 11.5 Estudio del número de generaciones
estudio_parametro("NUM_GENERACIONES", [10, 25, 50, 100], repeticiones=5)

In [ ]:
#@title 11.6 Con vs. sin elitismo
comparar_elitismo(repeticiones=10)

## 12. Cómo agregar lo que pida el doctor
- **Otra función:** escríbela en la celda 2 y agrégala al diccionario `FUNCIONES` (objetivo, intervalo, fórmula y óptimo si se conoce); después añade su clave a la lista del formulario de la celda 3.
- **Otro operador de cruce** (multipunto, uniforme…): escribe la función en la celda 7 con la misma forma que `cruce_monopunto(padre1, padre2)` y cámbiala en `reproduccion(...)` dentro de `algoritmo_genetico`.
- **Otra selección** (torneo, escalado…): escribe la función en la celda 6 con la misma forma que `seleccion_ruleta(pob)` y cámbiala dentro del bucle.
- **Otro criterio de terminación:** cambia el `for g in range(...)` por un `while` con la condición deseada.